# 01 - 向量嵌入模型 (Embedding Models)

## 学习目标

本notebook深入讲解RAG系统中的向量嵌入技术，包括：

1. **稠密嵌入 (Dense Embedding)**
   - Mean Pooling原理
   - L2归一化
   - 余弦相似度计算

2. **稀疏嵌入 (BM25)**
   - BM25算法详解
   - IDF和TF计算
   - 参数调优 (k1, b)

3. **混合嵌入 (Hybrid Embedding)**
   - 线性加权融合
   - RRF (Reciprocal Rank Fusion)

4. **向量相似度评估**
   - 批量相似度计算
   - 最近邻搜索
   - 可视化分析

---

## 0. 环境设置

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

from embeddings import (
    EmbeddingConfig,
    DenseEmbedding,
    SparseEmbedding,
    HybridEmbedding,
)

print("环境导入完成！")
print(f"NumPy版本: {np.__version__}")

## 1. 稠密嵌入 (Dense Embedding)

### 1.1 核心概念

**稠密嵌入**使用神经网络将文本映射到连续向量空间，使语义相似的文本在空间中距离相近。

**数学原理**:
```
E(text) = (1/n) * Σ E(token_i)  # Mean Pooling
E_norm = E / ||E||_2           # L2归一化
```

**优势**:
- 捕获语义相似性
- 适合同义词/近义词匹配
- 跨语言能力

**劣势**:
- 计算成本高
- 需要训练数据
- 精确关键词匹配弱

### 1.2 初始化稠密嵌入模型

In [ ]:
# 创建配置
config = EmbeddingConfig(
    dimension=256,        # 嵌入维度
    normalize=True,       # L2归一化
    batch_size=32,        # 批处理大小
    max_length=512,       # 最大序列长度
)

# 初始化稠密嵌入模型
dense = DenseEmbedding(config, random_seed=42)

print("稠密嵌入模型初始化完成！")
print(f"嵌入维度: {config.dimension}")
print(f"是否归一化: {config.normalize}")

### 1.3 单文本嵌入

In [ ]:
# 测试文本
text = "机器学习是人工智能的核心技术，通过数据训练模型实现预测和决策。"

# 生成嵌入向量
vector = dense.embed_text(text)

print(f"原始文本: {text}")
print(f"\n嵌入向量信息:")
print(f"  维度: {vector.shape}")
print(f"  数据类型: {vector.dtype}")
print(f"  范数: {np.linalg.norm(vector):.6f}")
print(f"  最小值: {vector.min():.4f}")
print(f"  最大值: {vector.max():.4f}")
print(f"  均值: {vector.mean():.4f}")
print(f"  标准差: {vector.std():.4f}")

### 1.4 批量嵌入

In [ ]:
# 批量文本
texts = [
    "深度学习使用多层神经网络进行特征学习",
    "自然语言处理让计算机理解人类语言",
    "计算机视觉使机器能够识别图像和视频",
    "强化学习通过奖励信号学习最优策略",
    "Transformer架构彻底改变了自然语言处理领域",
]

# 批量生成嵌入
vectors = dense.embed_texts(texts)

print(f"批量嵌入形状: {vectors.shape}")
print(f"\n每个文本的嵌入范数:")
for i, (text, vec) in enumerate(zip(texts, vectors)):
    norm = np.linalg.norm(vec)
    print(f"  [{i+1}] {norm:.6f} - {text[:30]}...")

### 1.5 余弦相似度计算

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """计算余弦相似度。
    
    cos(A,B) = (A·B) / (||A|| × ||B||)
    """
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def compute_similarity_matrix(vectors: np.ndarray, texts: List[str]) -> np.ndarray:
    """计算相似度矩阵。"""
    n = len(vectors)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            matrix[i, j] = cosine_similarity(vectors[i], vectors[j])
    return matrix

# 计算相似度矩阵
sim_matrix = compute_similarity_matrix(vectors, texts)

print("相似度矩阵:")
print("=" * 80)
for i, text in enumerate(texts):
    print(f"[{i+1}] {text[:35]:<35}", end="")
    for j in range(len(texts)):
        print(f" {sim_matrix[i, j]:5.3f}", end="")
    print()
print()

### 1.6 最近邻搜索

In [ ]:
def find_neighbors(query: str, model: DenseEmbedding, 
                  corpus: List[str], top_k: int = 3) -> List[Tuple[str, float]]:
    """查找最相似的top_k个文档。"""
    query_vec = model.embed_text(query)
    corpus_vecs = model.embed_texts(corpus)
    
    # 计算相似度
    similarities = [cosine_similarity(query_vec, vec) for vec in corpus_vecs]
    
    # 排序获取top_k
    indices = np.argsort(similarities)[::-1][:top_k]
    
    return [(corpus[i], similarities[i]) for i in indices]

# 测试查询
query = "神经网络和深度学习"
neighbors = find_neighbors(query, dense, texts, top_k=3)

print(f"查询: {query}")
print(f"\n最相关的3个文档:")
for i, (doc, score) in enumerate(neighbors, 1):
    print(f"  [{i}] 相似度: {score:.4f}")
    print(f"      内容: {doc}\n")

## 2. 稀疏嵌入 (BM25)

### 2.1 核心概念

**BM25 (Best Matching 25)** 是一种基于概率检索模型的排序函数。

**数学公式**:
```
score(D, Q) = Σ IDF(qi) × TF(qi, D)

IDF(q) = log((N - df(q) + 0.5) / (df(q) + 0.5) + 1)
TF(q, D) = f(q, D) × (k1 + 1) / (f(q, D) + k1 × (1 - b + b × |D|/avgdl))
```

**参数说明**:
- `k1`: 词频饱和参数 (1.2-2.0, 默认1.5)
- `b`: 长度归一化参数 (0-1, 默认0.75)

**调优指南**:
- `k1` 越大: 词频影响越大 (适合短查询)
- `b` 越大: 长文档惩罚越大 (适合长文档)

### 2.2 初始化BM25模型

In [ ]:
# 准备文档集合
documents = [
    "机器学习是人工智能的核心技术，通过数据训练模型实现预测和决策",
    "深度学习使用多层神经网络进行特征学习，在图像识别中表现出色",
    "自然语言处理让计算机理解人类语言，应用于翻译、问答和文本生成",
    "计算机视觉使机器能够识别图像和视频，用于人脸识别和物体检测",
    "强化学习通过奖励信号学习最优策略，广泛应用于游戏和机器人控制",
    "Transformer架构基于自注意力机制，彻底改变了自然语言处理领域",
    "卷积神经网络是深度学习的重要架构，特别适合处理图像数据",
    "循环神经网络擅长处理序列数据，但在长序列上存在梯度消失问题",
    "词嵌入将词语映射到向量空间，捕获词语之间的语义关系",
    "注意力机制让模型能够关注输入的重要部分，提高模型性能",
]

# 初始化BM25
sparse = SparseEmbedding(k1=1.5, b=0.75)
sparse.fit(documents)

print("BM25模型训练完成！")
print(f"文档数量: {len(documents)}")
print(f"词汇表大小: {sparse.vocab_size}")
print(f"平均文档长度: {sparse._avgdl:.2f}")

### 2.3 BM25分数计算

In [ ]:
query = "深度学习 神经网络"
query_vec = sparse.embed_text(query)

print(f"查询: {query}")
print(f"稀疏向量维度: {len(query_vec)}")
print(f"非零元素: {np.count_nonzero(query_vec)}")

# 计算所有文档的BM25分数
doc_vecs = sparse.embed_texts(documents)
scores = np.dot(doc_vecs, query_vec)

# 排序
sorted_indices = np.argsort(scores)[::-1]

print(f"\nTop-5最相关文档:")
for i, idx in enumerate(sorted_indices[:5], 1):
    print(f"  [{i}] BM25分数: {scores[idx]:.4f}")
    print(f"      文档: {documents[idx][:60]}...")

### 2.4 参数调优实验

In [ ]:
def evaluate_bm25_params(k1_values: List[float], b_values: List[float]) -> dict:
    """评估不同k1和b参数的效果。"""
    results = {}
    
    for k1 in k1_values:
        for b in b_values:
            model = SparseEmbedding(k1=k1, b=b)
            model.fit(documents)
            
            # 计算测试查询的分数
            query_vec = model.embed_text(query)
            doc_vecs = model.embed_texts(documents)
            scores = np.dot(doc_vecs, query_vec)
            
            results[(k1, b)] = {
                'max_score': scores.max(),
                'mean_score': scores.mean(),
                'top_doc_idx': np.argmax(scores),
            }
    
    return results

# 测试不同参数组合
k1_values = [0.5, 1.0, 1.5, 2.0]
b_values = [0.0, 0.5, 0.75, 1.0]

results = evaluate_bm25_params(k1_values, b_values)

print("参数调优结果:")
print(f"\n{'k1':>4} {'b':>4} {'Max Score':>10} {'Mean Score':>10} {'Top Doc':>8}")
print("-" * 42)
for (k1, b), res in sorted(results.items()):
    print(f"{k1:4.1f} {b:4.2f} {res['max_score']:10.4f} {res['mean_score']:10.4f} {res['top_doc_idx']:8d}")

## 3. 混合嵌入 (Hybrid Embedding)

### 3.1 核心概念

**混合嵌入**结合稠密嵌入和稀疏嵌入的优势。

**融合策略**:

1. **线性加权**:
```
hybrid_score = α × dense_score + (1-α) × sparse_score
```

2. **RRF (Reciprocal Rank Fusion)**:
```
rrf_score = Σ 1/(k + rank_i)  # k=60为推荐值
```

**最佳实践**:
- 语义搜索: α = 0.7-0.9
- 关键词搜索: α = 0.3-0.5
- 平衡搜索: α = 0.5-0.7

### 3.2 创建混合嵌入模型

In [ ]:
# 初始化稠密和稀疏嵌入
config = EmbeddingConfig(dimension=128, normalize=True)
dense_emb = DenseEmbedding(config, random_seed=42)
sparse_emb = SparseEmbedding(k1=1.5, b=0.75)
sparse_emb.fit(documents)

# 创建混合嵌入 (alpha=0.7偏向稠密)
hybrid = HybridEmbedding(
    dense_embedding=dense_emb,
    sparse_embedding=sparse_emb,
    alpha=0.7,
)

print("混合嵌入模型创建完成！")
print(f"稠密权重 (alpha): {hybrid.alpha}")
print(f"稀疏权重 (1-alpha): {1 - hybrid.alpha}")
print(f"混合向量维度: {config.dimension + sparse_emb.vocab_size}")

### 3.3 混合嵌入检索

In [ ]:
test_query = "深度学习"
hybrid_vec = hybrid.embed_text(test_query)

print(f"查询: {test_query}")
print(f"混合向量维度: {len(hybrid_vec)}")
print(f"  - 稠密部分: {config.dimension}维")
print(f"  - 稀疏部分: {sparse_emb.vocab_size}维")

# 获取单独的稠密和稀疏向量
dense_vec = hybrid.get_dense_embedding(test_query)
sparse_vec = hybrid.get_sparse_embedding(test_query)

print(f"\n稠密向量范数: {np.linalg.norm(dense_vec):.6f}")
print(f"稀疏向量非零元素: {np.count_nonzero(sparse_vec)}")

### 3.4 Alpha参数影响分析

In [ ]:
def compare_alpha_values(alpha_values: List[float]) -> None:
    """比较不同alpha值的检索效果。"""
    test_queries = [
        "深度学习",
        "神经网络",
        "Transformer",
    ]
    
    for alpha in alpha_values:
        print(f"\n{'='*60}")
        print(f"Alpha = {alpha:.1f} (稠密权重: {alpha:.0%}, 稀疏权重: {1-alpha:.0%})")
        print('='*60)
        
        # 创建混合嵌入
        h = HybridEmbedding(dense_emb, sparse_emb, alpha=alpha)
        
        for query in test_queries:
            # 计算相似度
            query_vec = h.embed_text(query)
            doc_vecs = h.embed_texts(documents)
            sims = [cosine_similarity(query_vec, dv) for dv in doc_vecs]
            
            # Top-3
            top_indices = np.argsort(sims)[::-1][:3]
            
            print(f"\n查询: {query}")
            for i, idx in enumerate(top_indices, 1):
                print(f"  [{i}] {sims[idx]:.4f} - {documents[idx][:50]}...")

# 测试不同alpha值
alpha_values = [0.0, 0.3, 0.5, 0.7, 1.0]
compare_alpha_values(alpha_values)

## 4. 向量嵌入可视化

In [ ]:
# 使用降维技术可视化高维向量
from sklearn.decomposition import PCA

# 生成2D嵌入用于可视化
vis_config = EmbeddingConfig(dimension=50, normalize=True)
vis_dense = DenseEmbedding(vis_config, random_seed=42)

# 获取向量
sample_texts = documents[:6]
vectors_50d = vis_dense.embed_texts(sample_texts)

# 降维到2D
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors_50d)

print("降维结果:")
print(f"原始维度: {vectors_50d.shape[1]}")
print(f"降维后: {vectors_2d.shape[1]}")
print(f"解释方差比: {pca.explained_variance_ratio_.sum():.2%}")

# 可视化
plt.figure(figsize=(10, 6))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], s=100, alpha=0.6)

for i, text in enumerate(sample_texts):
    plt.annotate(text[:15] + "...", 
                 (vectors_2d[i, 0], vectors_2d[i, 1]),
                 fontsize=8,
                 xytext=(5, 5),
                 textcoords='offset points')

plt.title('文本嵌入向量2D可视化 (PCA降维)', fontsize=14)
plt.xlabel('第一主成分', fontsize=12)
plt.ylabel('第二主成分', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n文本向量可视化完成！")

## 5. 总结

本notebook涵盖了向量嵌入的核心技术：

1. **稠密嵌入**: 使用神经网络捕获语义相似性
2. **稀疏嵌入 (BM25)**: 基于词频的精确匹配
3. **混合嵌入**: 平衡语义和关键词匹配
4. **可视化**: 理解高维向量空间

### 关键要点

- **Mean Pooling**: 简单有效的文本表示方法
- **L2归一化**: 确保向量在单位超球面上
- **余弦相似度**: 衡量语义相似度的标准方法
- **BM25参数**: k1控制词频饱和，b控制长度惩罚
- **混合权重**: alpha根据应用场景调整

### 下一步

- 学习向量数据库 (02_vector_databases.ipynb)
- 了解检索策略优化
- 实现完整RAG流水线